# Enterprise-wide monitoring of Azure AI Foundry & Azure OpenAI

This notebook collects quota, deployment, and usage data across **all Azure subscriptions** accessible to the current credential, then visualizes utilization.

**Data collected (one table per step):**

1. **Subscriptions** — list all enabled Azure subscriptions
2. **Instances** — discover every Cognitive Services instance (Azure AI Foundry + Azure OpenAI) across all subscriptions
3. **`subscriptionQuota`** — subscription-level model quota: one row per *subscription × region × model*, showing how much TPM capacity is allocated to deployments vs. the subscription maximum
   - this is only available through a call to ARM
4. **`deploymentQuota`** — deployment-level rate limits: one row per *instance × deployment*, with SKU, capacity, TPM limit, and RPM limit
   - this is only available through a call to ARM
5. **`tokenUsageOverTime`** — hourly token usage timeseries: one row per *instance × deployment × hour*, with prompt, generated, and total token counts
   - this comes from Azure Monitor but is not available in log analytics with a per-model detail when diagnostic settings stream the data

In [ ]:
%pip install azure-identity azure-monitor-query azure-monitor-querymetrics requests pandas plotly --quiet

In [ ]:
"""
Configuration — adjust these values for your environment.
"""
from datetime import timedelta

from azure.identity import DefaultAzureCredential
import requests
import pandas as pd

# ── Azure credential (uses az login / managed identity / env vars) ──
credential = DefaultAzureCredential()

# ── ARM API version for Cognitive Services ──────────────────────────
API_VERSION = "2024-10-01"

# ── Token usage lookback period ─────────────────────────────────────
TOKEN_USAGE_LOOKBACK = timedelta(hours=24)
TOKEN_USAGE_GRANULARITY = timedelta(hours=1)

## Step 1 — List all enabled Azure subscriptions

Queries the ARM API for every subscription the current credential can access and filters to enabled ones.

In [ ]:
token = credential.get_token("https://management.azure.com/.default").token
headers = {"Authorization": f"Bearer {token}"}

subs_url = "https://management.azure.com/subscriptions?api-version=2022-12-01"
subs_resp = requests.get(subs_url, headers=headers)
subs_resp.raise_for_status()

subscriptions = [
    {"subscriptionId": s["subscriptionId"], "displayName": s["displayName"]}
    for s in subs_resp.json()["value"]
    if s.get("state") == "Enabled"
]

df_subscriptions = pd.DataFrame(subscriptions)
print(f"Found {len(df_subscriptions)} enabled subscriptions")
display(df_subscriptions)

## Step 2 — Discover all Foundry & Azure OpenAI instances (across all subscriptions)

Lists every `Microsoft.CognitiveServices/accounts` resource across all subscriptions. This covers both **Azure AI Foundry** (`kind: AIServices`) and **Azure OpenAI** (`kind: OpenAI`). Result: one row per instance.

In [ ]:
# Discover every Microsoft.CognitiveServices/accounts resource across all subscriptions.
# Covers Azure AI Foundry (kind: AIServices) and Azure OpenAI (kind: OpenAI). One row per instance.
instances = []

for sub in subscriptions:
    sub_id = sub["subscriptionId"]
    accounts_url = (
        f"https://management.azure.com/subscriptions/{sub_id}"
        f"/providers/Microsoft.CognitiveServices/accounts"
        f"?api-version={API_VERSION}"
    )
    resp = requests.get(accounts_url, headers=headers)
    if resp.status_code != 200:
        print(f"  ⚠ Could not list accounts in {sub['displayName']}: {resp.status_code}")
        continue
    for acct in resp.json().get("value", []):
        instances.append({
            "subscriptionId": sub_id,
            "subscriptionName": sub["displayName"],
            "resourceGroup": acct["id"].split("/")[4],
            "name": acct["name"],
            "location": acct["location"],
            "kind": acct.get("kind", ""),
            "resourceId": acct["id"],
        })

df_instances = pd.DataFrame(instances)
print(f"Found {len(df_instances)} Cognitive Services instances across {len(subscriptions)} subscriptions")
display(df_instances[["subscriptionName", "name", "location", "kind"]])

## Step 3 — Subscription-level model quota (per subscription × region × model)

Queries the ARM usages API for each unique subscription + region pair to get model-level TPM quota.

- **`currentValue`** = total TPM capacity currently allocated to deployments under this subscription/region/model
- **`limit`** = maximum TPM capacity allowed by the subscription for this model in this region

Result: one row per *subscription × region × model*.

In [ ]:
# Build unique (subscriptionId, location) pairs from discovered instances
sub_location_pairs = (
    df_instances[["subscriptionId", "location"]]
    .drop_duplicates()
    .values.tolist()
)

quota_rows = []
for sub_id, location in sub_location_pairs:
    usages_url = (
        f"https://management.azure.com/subscriptions/{sub_id}"
        f"/providers/Microsoft.CognitiveServices/locations/{location}"
        f"/usages?api-version={API_VERSION}"
    )
    resp = requests.get(usages_url, headers=headers)
    if resp.status_code != 200:
        print(f"  ⚠ Could not get quota for {sub_id} / {location}: {resp.status_code}")
        continue
    for u in resp.json().get("value", []):
        quota_rows.append({
            "subscriptionId": sub_id,
            "location": location,
            "modelName": u["name"]["value"],
            "currentValue": u["currentValue"],
            "limit": u["limit"],
        })

subscriptionQuota = pd.DataFrame(quota_rows)

# Show only models that have some allocation or a non-zero limit
subscriptionQuota_active = subscriptionQuota[
    (subscriptionQuota["currentValue"] > 0) | (subscriptionQuota["limit"] > 0)
].copy()

print(f"subscriptionQuota: {len(subscriptionQuota_active)} model quotas with allocation across {len(sub_location_pairs)} subscription/region pairs")
display(subscriptionQuota_active)

## Step 4 — Deployment-level rate limits (per instance × deployment)

Lists every model deployment across all discovered instances, including SKU tier, provisioned capacity, and configured TPM / RPM rate limits.

Result: one row per *instance × deployment*.

In [ ]:
deploy_rows = []

for _, inst in df_instances.iterrows():
    deploys_url = (
        f"https://management.azure.com{inst['resourceId']}"
        f"/deployments?api-version={API_VERSION}"
    )
    resp = requests.get(deploys_url, headers=headers)
    if resp.status_code != 200:
        print(f"  ⚠ Could not list deployments for {inst['name']}: {resp.status_code}")
        continue
    for d in resp.json().get("value", []):
        props = d.get("properties", {})
        model = props.get("model", {})
        sku = d.get("sku", {})
        rate_limits = props.get("rateLimits", [])
        tpm = next((r["count"] for r in rate_limits if r.get("key") == "token"), None)
        rpm = next((r["count"] for r in rate_limits if r.get("key") == "request"), None)
        # Fall back to SKU capacity (in K) converted to tokens if rateLimits not present
        # if tpm is None and sku.get("capacity"):
        #     tpm = int(sku["capacity"]) * 1000
        deploy_rows.append({
            "subscriptionId": inst["subscriptionId"],
            "location": inst["location"],
            "instanceName": inst["name"],
            "modelName": model.get("name", ""),
            "deploymentName": d["name"],
            "sku": sku.get("name", ""),
            "capacity": sku.get("capacity", ""),
            "tpmLimit": tpm,
            "rpmLimit": rpm,
        })

deploymentQuota = pd.DataFrame(deploy_rows)
print(f"deploymentQuota: {len(deploymentQuota)} deployments across {df_instances['name'].nunique()} instances")
display(deploymentQuota)

## Step 5 — Hourly token usage timeseries (per instance × deployment × hour)

Queries Azure Monitor Metrics for each instance, fetching `ProcessedPromptTokens` and `GeneratedTokens` split by deployment name over the configured lookback period.

Result: one row per *instance × deployment × hour* containing prompt, generated, and total token counts.

In [ ]:
from azure.monitor.querymetrics import MetricsClient, MetricAggregationType

# Build a deployment-name → model-name lookup from deploymentQuota
deployment_model_map = {}
for _, row in deploymentQuota.iterrows():
    key = (row["instanceName"], row["deploymentName"])
    deployment_model_map[key] = row["modelName"]

token_rows = []
_metrics_clients = {}  # cache MetricsClient per region

for _, inst in df_instances.iterrows():
    location = inst["location"]
    if location not in _metrics_clients:
        endpoint = f"https://{location}.metrics.monitor.azure.com"
        _metrics_clients[location] = MetricsClient(endpoint=endpoint, credential=credential)

    metrics_client = _metrics_clients[location]
    resource_id = inst["resourceId"]

    try:
        results = metrics_client.query_resources(
            resource_ids=[resource_id],
            metric_namespace="Microsoft.CognitiveServices/accounts",
            metric_names=["ProcessedPromptTokens", "GeneratedTokens"],
            timespan=TOKEN_USAGE_LOOKBACK,
            granularity=TOKEN_USAGE_GRANULARITY,
            aggregations=[MetricAggregationType.TOTAL],
            filter="ModelDeploymentName eq '*'",
        )
    except Exception as e:
        print(f"  ⚠ Metrics query failed for {inst['name']}: {e}")
        continue

    # Collect per-metric data keyed by (deployment, timestamp)
    metric_data = {}  # (deployment, timestamp) → {prompt: x, generated: y}
    for result in results:
        for metric in result.metrics:
            for ts_element in metric.timeseries:
                deployment = ts_element.metadata_values.get("modeldeploymentname", "")
                for dp in ts_element.data:
                    if dp.total is None:
                        continue
                    key = (deployment, dp.timestamp)
                    if key not in metric_data:
                        metric_data[key] = {"promptTokens": 0, "generatedTokens": 0}
                    if metric.name == "ProcessedPromptTokens":
                        metric_data[key]["promptTokens"] = dp.total
                    elif metric.name == "GeneratedTokens":
                        metric_data[key]["generatedTokens"] = dp.total

    for (deployment, timestamp), values in metric_data.items():
        model_name = deployment_model_map.get((inst["name"], deployment), "")
        token_rows.append({
            "subscriptionId": inst["subscriptionId"],
            "location": inst["location"],
            "instanceName": inst["name"],
            "modelName": model_name,
            "deploymentName": deployment,
            "timestamp": timestamp,
            "promptTokens": values["promptTokens"],
            "generatedTokens": values["generatedTokens"],
            "totalTokens": values["promptTokens"] + values["generatedTokens"],
        })

# tokenUsageOverTime: one row per (instance, deployment, hour) with prompt/generated/total token counts
tokenUsageOverTime = pd.DataFrame(token_rows)
if not tokenUsageOverTime.empty:
    tokenUsageOverTime = tokenUsageOverTime.sort_values(["instanceName", "deploymentName", "timestamp"]).reset_index(drop=True)

print(f"tokenUsage: {len(tokenUsageOverTime)} rows across {df_instances['name'].nunique()} instances")
display(tokenUsageOverTime)

## Visualizations

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

### Subscription-level quota utilization (per model per region)

Shows what percentage of each subscription's per-model TPM limit is already allocated to deployments. One horizontal bar per *model × region*; red bars indicate ≥ 80 % utilization.

In [ ]:
sq = subscriptionQuota_active.copy()
sq = sq[sq["limit"] > 0].copy()
sq["utilizationPct"] = (sq["currentValue"] / sq["limit"] * 100).round(1)
sq["label"] = sq["modelName"] + " (" + sq["location"] + ")"
sq = sq.sort_values("utilizationPct", ascending=True)

fig = go.Figure()
fig.add_trace(go.Bar(
    y=sq["label"],
    x=sq["utilizationPct"],
    orientation="h",
    marker_color=["#EF553B" if v >= 80 else "#636EFA" for v in sq["utilizationPct"]],
    text=[f"{v}% ({int(c)}/{int(l)})" for v, c, l in zip(sq["utilizationPct"], sq["currentValue"], sq["limit"])],
    textposition="outside",
))
fig.add_vline(x=100, line_dash="dash", line_color="red", annotation_text="Limit")
fig.update_layout(
    title="Subscription Quota Allocated vs Limit (per model, per region)",
    xaxis_title="Allocated %",
    height=500,
    margin=dict(l=200),
    showlegend=False,
)
fig.show()

### Token usage over time & deployment-level rate-limit utilization

**Chart 1 — Tokens consumed per hour (per deployment):** One line per deployment showing total tokens (prompt + generated) consumed each hour.

**Chart 2 — Rate-limit utilization % over time (per deployment):** Converts hourly token totals to an average TPM and divides by the deployment's configured TPM limit. The red dashed line marks 100 %.

In [ ]:
if not tokenUsageOverTime.empty:
    tu = tokenUsageOverTime.copy()
    tu["deployment_label"] = tu["instanceName"] + " / " + tu["deploymentName"]

    # ── Token usage over time ───────────────────────────────────────
    fig1 = px.line(
        tu,
        x="timestamp",
        y="totalTokens",
        color="deployment_label",
        title="Tokens Consumed per Hour (per deployment)",
        labels={"totalTokens": "Total Tokens", "timestamp": "Time", "deployment_label": "Deployment"},
    )
    fig1.update_layout(legend=dict(orientation="h", yanchor="top", y=-0.25))
    fig1.show()

    # ── Utilization % over time ─────────────────────────────────────
    # Join with deploymentQuota to get tpmLimit per deployment
    dq_lookup = deploymentQuota[["instanceName", "deploymentName", "tpmLimit"]].copy()
    tu_util = tu.merge(dq_lookup, on=["instanceName", "deploymentName"], how="left")
    tu_util = tu_util[tu_util["tpmLimit"].notna() & (tu_util["tpmLimit"] > 0)].copy()

    if not tu_util.empty:
        # TPM limit is per-minute; hourly tokens / 60 gives avg tokens-per-minute
        tu_util["utilizationPct"] = (tu_util["totalTokens"] / 60) / tu_util["tpmLimit"] * 100

        fig2 = px.line(
            tu_util,
            x="timestamp",
            y="utilizationPct",
            color="deployment_label",
            title="Deployment Rate-Limit Utilization % Over Time (avg TPM vs deployment TPM limit)",
            labels={"utilizationPct": "Rate-Limit Utilization %", "timestamp": "Time", "deployment_label": "Deployment"},
        )
        max_util = tu_util["utilizationPct"].max()
        if max_util < 100:
            fig2.update_yaxes(range=[0, max(max_util * 1.2, 1)])
        fig2.add_hline(y=100, line_dash="dash", line_color="red", annotation_text="100% limit")
        fig2.update_layout(
            yaxis_ticksuffix="%",
            legend=dict(orientation="h", yanchor="top", y=-0.25),
        )
        fig2.show()
    else:
        print("⚠ No deployments with TPM limits found — cannot compute utilization %")
else:
    print("⚠ No token usage data — skipping charts")

### Peak actual TPM vs configured rate limit (per deployment)

Overlaid horizontal bars comparing each deployment's **peak observed TPM** (highest single hour ÷ 60) against its **configured TPM rate limit**. Helps identify deployments that are near capacity or over-provisioned.

In [ ]:
if not tokenUsageOverTime.empty and not deploymentQuota.empty:
    # Peak hourly tokens ÷ 60 → peak tokens-per-minute per deployment
    max_tpm = (
        tokenUsageOverTime.groupby(["instanceName", "deploymentName"])["totalTokens"]
        .max()
        .reset_index()
        .rename(columns={"totalTokens": "maxTokensPerHour"})
    )
    max_tpm["maxTPM"] = max_tpm["maxTokensPerHour"] / 60

    cap = max_tpm.merge(
        deploymentQuota[["instanceName", "deploymentName", "tpmLimit"]],
        on=["instanceName", "deploymentName"],
        how="left",
    )
    cap = cap[cap["tpmLimit"].notna() & (cap["tpmLimit"] > 0)].copy()

    if not cap.empty:
        cap["label"] = cap["instanceName"] + " / " + cap["deploymentName"]
        cap = cap.sort_values("maxTPM", ascending=True)

        fig = go.Figure()
        fig.add_trace(go.Bar(
            y=cap["label"], x=cap["tpmLimit"], name="TPM Limit",
            orientation="h", marker_color="#EF553B", opacity=0.4,
        ))
        fig.add_trace(go.Bar(
            y=cap["label"], x=cap["maxTPM"], name="Max Actual TPM",
            orientation="h", marker_color="#636EFA",
        ))
        fig.update_layout(
            barmode="overlay",
            title="Peak Actual TPM vs Deployment Rate Limit",
            xaxis_title="Tokens per Minute",
            height=max(400, len(cap) * 35),
            legend=dict(orientation="h", yanchor="top", y=-0.15),
        )
        fig.show()
    else:
        print("⚠ No deployments with TPM limits — cannot show utilization")
else:
    print("⚠ Missing data — skipping chart")

### Total token consumption ranking (per deployment)

Stacked bar chart ranking each deployment by total tokens consumed over the lookback period. Bars are split into **prompt (input)** and **generated (output)** tokens to show the input/output ratio.

In [ ]:
if not tokenUsageOverTime.empty:
    top = (
        tokenUsageOverTime.groupby(["instanceName", "deploymentName", "modelName"])
        .agg(totalTokens=("totalTokens", "sum"), promptTokens=("promptTokens", "sum"), generatedTokens=("generatedTokens", "sum"))
        .reset_index()
        .sort_values("totalTokens", ascending=True)
    )
    top["label"] = top["instanceName"] + " / " + top["deploymentName"]

    fig = go.Figure()
    fig.add_trace(go.Bar(
        y=top["label"], x=top["promptTokens"], name="Prompt Tokens",
        orientation="h", marker_color="#636EFA",
    ))
    fig.add_trace(go.Bar(
        y=top["label"], x=top["generatedTokens"], name="Generated Tokens",
        orientation="h", marker_color="#00CC96",
    ))
    fig.update_layout(
        barmode="stack",
        title=f"Top Deployments by Total Token Consumption (last {TOKEN_USAGE_LOOKBACK})",
        xaxis_title="Total Tokens",
        height=max(400, len(top) * 35),
        legend=dict(orientation="h", yanchor="top", y=-0.15),
    )
    fig.show()
else:
    print("⚠ No token usage data — skipping chart")